In [17]:
import requests
import time
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

url = "https://www.flipkart.com/search?q=laptop&sid=6bo%2Cb5g&as=on&as-show=on&otracker=AS_QueryStore_OrganicAutoSuggest_1_3_na_na_na&otracker1=AS_QueryStore_OrganicAutoSuggest_1_3_na_na_na&as-pos=1&as-type=RECENT&suggestionId=laptop%7CLaptops&requestId=1ae2efb7-0f44-49d3-a117-9abab6674812&as-searchtext=laptop&page={page}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36"
}

brand = []
model = []
price = []
rating = []
processor = []
ram = []
rom = []

for page in range(1, 31):
    response = requests.get(
        url.format(page=page),
        headers=headers,
        timeout=20
    )

    print("Page:", page, "Status:", response.status_code)

    if response.status_code != 200:
        print("Page request failed")
        break

    soup = BeautifulSoup(response.text, "html.parser")

    information = soup.find_all("div", class_="RG5Slk")
    p = soup.find_all("div", class_="CMXw7N")
    price_tags = soup.find_all("div", class_="oFEPlD")
    rating_tags = soup.find_all("div", class_="a7saXW")

    if not information:
        print("No laptop names found. Check the returned HTML.")
        break

    for index, item in enumerate(information):
        name = item.get_text(strip=True)

        if not name:
            continue

        brand.append(name.split()[0])
        model.append(name.split(" - ")[0])

        product_price = np.nan
        product_rating = np.nan
        product_processor = np.nan
        product_ram = np.nan
        product_rom = np.nan

        if index < len(price_tags):
            product_price = price_tags[index].get_text(strip=True)

        if index < len(rating_tags):
            product_rating = rating_tags[index].get_text(strip=True)

        if index < len(p):
            details = p[index].find_all("li", class_="DTBslk")
            specs = [d.get_text(strip=True) for d in details]

            if specs:
                product_processor = specs[0]

            for spec in specs:
                if "RAM" in spec.upper():
                    product_ram = spec

                if any(storage in spec.upper() for storage in ["SSD", "HDD", "EMMC"]):
                    product_rom = spec

        price.append(product_price)
        rating.append(product_rating)
        processor.append(product_processor)
        ram.append(product_ram)
        rom.append(product_rom)

        if len(brand) >= 500:
            break

    print("Records collected:", len(brand))

    if len(brand) >= 500:
        break

    time.sleep(2)

Page: 1 Status: 200
Records collected: 24
Page: 2 Status: 200
Records collected: 48
Page: 3 Status: 200
Records collected: 72
Page: 4 Status: 200
Records collected: 96
Page: 5 Status: 200
Records collected: 120
Page: 6 Status: 200
Records collected: 144
Page: 7 Status: 200
Records collected: 168
Page: 8 Status: 200
Records collected: 192
Page: 9 Status: 200
Records collected: 216
Page: 10 Status: 200
Records collected: 240
Page: 11 Status: 200
Records collected: 264
Page: 12 Status: 200
Records collected: 288
Page: 13 Status: 200
Records collected: 312
Page: 14 Status: 200
Records collected: 336
Page: 15 Status: 200
Records collected: 360
Page: 16 Status: 200
Records collected: 384
Page: 17 Status: 200
Records collected: 408
Page: 18 Status: 200
Records collected: 432
Page: 19 Status: 200
Records collected: 456
Page: 20 Status: 200
Records collected: 480
Page: 21 Status: 200
Records collected: 500


In [18]:
df = pd.DataFrame({
    "Brand": brand,
    "Model": model,
    "Price": price,
    "Rating": rating,
    "Processor": processor,
    "RAM": ram,
    "ROM": rom
})

In [19]:
df.to_csv("flipkart_laptops.csv", index=False)

In [20]:
print("Total records:", len(df))
print("Missing values:\n", df.isnull().sum())

Total records: 500
Missing values:
 Brand         0
Model         0
Price         0
Rating       97
Processor     0
RAM           0
ROM          15
dtype: int64
